In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("olist_pipeline")

In [2]:
logger.info("Starting data processing")

try:
    # Your processing code
    logger.info("Data processed successfully")

except Exception:
    logger.exception("Data processing failed")

2026-11-23 15:02:57,759 | INFO | olist_pipeline | Starting data processing
2026-11-23 15:02:57,762 | INFO | olist_pipeline | Data processed successfully


In [5]:
from datetime import datetime
import logging
from pathlib import Path

log_filename = f"logs/pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"

Path("logs").mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("olist_pipeline")

logger.info("Pipeline started")

2026-11-23 15:06:40,777 | INFO | olist_pipeline | Pipeline started


In [7]:
import yaml

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config)

{'paths': {'input': 'olist', 'output': 'output', 'logs': 'logs'}, 'thresholds': {'min_order_value': 0, 'max_missing_percentage': 20}, 'mappings': {'state_column': 'customer_state', 'city_column': 'customer_city'}}


In [10]:
import pandas as pd

input_path = Path(config["paths"]["input"])

df = pd.read_csv(input_path / "olist_orders_dataset.csv")

logger.info(f"Loaded {len(df)} records")
display(df.head())

2026-11-23 15:17:30,402 | INFO | olist_pipeline | Loaded 99441 records


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
from pathlib import Path


output_dir = Path(config["paths"]["output"])
output_dir.mkdir(parents=True, exist_ok=True)

# CSV output
df.to_csv(output_dir / "cleaned_data.csv", index=False)

# JSON output
df.to_json(
    output_dir / "cleaned_data.json",
    orient="records",
    indent=4
)

logger.info("CSV and JSON outputs saved successfully")

2026-11-23 15:17:34,005 | INFO | olist_pipeline | CSV and JSON outputs saved successfully


In [12]:
import json
from datetime import datetime

summary = {
    "run_timestamp": datetime.now().isoformat(),
    "input_file": "olist_orders_dataset.csv",
    "total_records": len(df),
    "total_columns": len(df.columns),
    "output_files": [
        "cleaned_data.csv",
        "cleaned_data.json"
    ],
    "status": "SUCCESS"
}

summary_path = output_dir / "run_summary.json"

with open(summary_path, "w") as file:
    json.dump(summary, file, indent=4)

logger.info("Run summary generated successfully")

print(summary)

2026-11-23 15:22:31,832 | INFO | olist_pipeline | Run summary generated successfully


{'run_timestamp': '2026-11-23T15:22:31.829456', 'input_file': 'olist_orders_dataset.csv', 'total_records': 99441, 'total_columns': 8, 'output_files': ['cleaned_data.csv', 'cleaned_data.json'], 'status': 'SUCCESS'}


In [32]:
import hashlib
import json

manifest_path = output_dir / "manifest.json"

def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


input_file = input_path / "olist_orders_dataset.csv"
current_hash = get_file_hash(input_file)

if manifest_path.exists():
    with open(manifest_path, "r") as file:
        manifest = json.load(file)
else:
    manifest = {}

if manifest.get(input_file.name) == current_hash:
    logger.info("File already processed. Skipping.")
else:
    df = pd.read_csv(input_file)

    df.to_csv(output_dir / "cleaned_data.csv", index=False)
    df.to_json(output_dir / "cleaned_data.json", orient="records", indent=4)

    manifest[input_file.name] = current_hash

    with open(manifest_path, "w") as file:
        json.dump(manifest, file, indent=4)

    logger.info("New or modified file processed successfully")

2026-11-23 15:43:00,369 | INFO | File already processed. Skipping.


In [35]:
import pandas as pd
import json
import hashlib
import logging
import yaml

from pathlib import Path
from datetime import datetime

# Load configuration
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

input_path = Path(config["paths"]["input"])
output_dir = Path(config["paths"]["output"])
log_dir = Path(config["paths"]["logs"])

output_dir.mkdir(exist_ok=True)
log_dir.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f"pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("pipeline")

# Input file
input_file = input_path / "olist_orders_dataset.csv"

# Generate file hash
def get_file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

manifest_path = output_dir / "manifest.json"

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = {}

file_hash = get_file_hash(input_file)

if manifest.get(input_file.name) == file_hash:
    logger.info("File already processed. Skipping pipeline.")

else:
    logger.info("Reading input file")

    df = pd.read_csv(input_file)

    df.to_csv(output_dir / "cleaned_data.csv", index=False)
    df.to_json(
        output_dir / "cleaned_data.json",
        orient="records",
        indent=4
    )

    summary = {
        "run_timestamp": datetime.now().isoformat(),
        "input_file": input_file.name,
        "total_records": len(df),
        "total_columns": len(df.columns),
        "status": "SUCCESS"
    }

    (output_dir / "run_summary.json").write_text(
        json.dumps(summary, indent=4)
    )

    manifest[input_file.name] = file_hash
    manifest_path.write_text(json.dumps(manifest, indent=4))

    logger.info("Pipeline completed successfully")

2026-11-23 15:44:56,255 | INFO | File already processed. Skipping pipeline.


In [30]:
!ruff check . --fix

E722 Do not use bare `except`
  --> day11.ipynb:cell 7:9:1
   |
 7 | try:
 8 |     df = pd.read_csv(input_file, sep=",")
 9 | except:
   | ^^^^^^
10 |     df = pd.read_csv(input_file, sep="\t")
   |

invalid-syntax: Unexpected indentation
 --> day13.ipynb:cell 16:2:1
  |
1 |         # 4. Business rule: price must be greater than 0
2 |         if "price" in df.columns and pd.notna(row["price"]):
  | ^^^^^^^^
3 |
4 |             try:
  |

invalid-syntax: Unexpected indentation
 --> day13.ipynb:cell 21:2:1
  |
1 |         # 6. ID regex validation
2 |         for column in ID_COLUMNS.get(filename, []):
  | ^^^^^^^^
3 |
4 |             value = row[column]
  |

invalid-syntax: Simple statements must be separated by newlines or semicolons
 --> day14.ipynb:cell 11:1:6
  |
1 | ruff check . --fix
  |      ^^^^^

invalid-syntax: Expected an identifier
 --> day14.ipynb:cell 11:1:14
  |
1 | ruff check . --fix
  |              ^

Found 6 errors (1 fixed, 5 remaining).


In [31]:
!ruff check *.py

All checks passed!


In [22]:
ruff check . --fix

SyntaxError: invalid syntax (140294464.py, line 1)

In [36]:
import json

with open("output/manifest.json", "r") as file:
    manifest_after = json.load(file)

print(manifest_after)

{'olist_orders_dataset.csv': '024eb3b79b8d87aba15560fa0718ae4c'}


In [37]:
import json

with open("output/manifest.json", "r") as file:
    manifest_after = json.load(file)

print(manifest_after)

{'olist_orders_dataset.csv': '024eb3b79b8d87aba15560fa0718ae4c'}
